# GenAI Pipeline — Batch Input

Reads publications from `publications.db`, builds Anthropic Message Batch requests, submits
the batch, and saves metadata to `batch_jobs/` for retrieval in `genai_scope_batchoutput.ipynb`.

Batch API gives a 50% discount versus standard API pricing and runs asynchronously (results
ready within 24 hours, usually much faster).

### 1. Imports and Configuration

In [ ]:
import duckdb
import pandas as pd
import json
import anthropic
import os
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

DB_PATH = "../publications.db"
BATCH_DIR = Path("batch_jobs")

In [ ]:
PROMPT_PATH = "1_prompt_debugging/v10/prompt_v10.md"

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Batch API gives 50% off all input/output token prices.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $0.50         $2.50       200K
# claude-sonnet-4-6        $1.50         $7.50       1M
# claude-opus-4-8          $2.50        $12.50       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["haiku"]  # ← change this to switch model

MAX_TOKENS = 512
TEMPERATURE = 0.0


###########################################################################
# Reasoning is useful for debugging prompt quality but costs extra tokens.
# Set False for large production runs to reduce token usage.
INCLUDE_REASONING = True
############################################################################


client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

### 2. Data Loading

In [ ]:
con = duckdb.connect(DB_PATH, read_only=True)
df = con.sql("SELECT * FROM publications_raw").df()
con.close()

print(f"Shape: {df.shape}")
print(f"\nScope distribution:\n{df['scope'].value_counts()}")
print(f"\nPillar distribution:\n{df['pillar'].value_counts()}")
df.head()

### 3. Dataset Selection

Choose which records to send. For a full production run use the entire `df`.
Filters are commented out below as examples.

In [ ]:
DATASET = df  # full dataset

# To process only records not yet labelled by LLM (if you add an llm_scope column later):
# DATASET = df[df["llm_scope"].isna()].reset_index(drop=True)

# To process a specific list of IDs:
# ids = ["pub.123", "pub.456"]
# DATASET = df[df["id"].isin(ids)].reset_index(drop=True)

print(f"Records to send: {len(DATASET)}")
DATASET[["id", "scope", "pillar"]].head()

### 4. Load Prompt

In [ ]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

system_prompt = load_prompt()
print(system_prompt)

### 5. Build Batch Requests

The Batch API uses standard message parameters — structured output is achieved via
tool use with `tool_choice` forced to the classification tool, mirroring the
Pydantic schema used in `genai_testing.ipynb`.

In [ ]:
_TOOL_PROPERTIES = {
    "scope": {
        "type": "string",
        "enum": ["in", "out"],
        "description": "Whether the publication is in scope for alternative proteins."
    },
    "confidence": {
        "type": "integer",
        "minimum": 1,
        "maximum": 5,
        "description": "Confidence score 1-5 for the scope decision."
    },
    "plant_based": {"type": "boolean"},
    "fermentation": {"type": "boolean"},
    "cultivated": {"type": "boolean"},
    "cross_cutting": {"type": "boolean"},
    "reasoning": {
        "type": "string",
        "description": "One concise sentence explaining the scope decision."
    },
}
_REQUIRED_BASE = ["scope", "confidence", "plant_based", "fermentation", "cultivated", "cross_cutting"]

def _build_tool():
    props = {k: v for k, v in _TOOL_PROPERTIES.items() if k != "reasoning" or INCLUDE_REASONING}
    required = _REQUIRED_BASE + (["reasoning"] if INCLUDE_REASONING else [])
    return {
        "name": "classify_publication",
        "description": "Record the scope and pillar classification for a research publication.",
        "input_schema": {
            "type": "object",
            "properties": props,
            "required": required,
        }
    }

CLASSIFICATION_TOOL = _build_tool()


def build_batch_request(row):
    user_message = f"Title: {row['title']}\n\nAbstract: {row['abstract']}"
    return {
        "custom_id": row["id"],
        "params": {
            "model": MODEL,
            "max_tokens": MAX_TOKENS,
            "temperature": TEMPERATURE,
            "system": [
                {
                    "type": "text",
                    "text": system_prompt,
                    "cache_control": {"type": "ephemeral"}
                }
            ],
            "messages": [{"role": "user", "content": user_message}],
            "tools": [CLASSIFICATION_TOOL],
            "tool_choice": {"type": "tool", "name": "classify_publication"},
        }
    }

In [ ]:
batch_requests = [build_batch_request(row) for _, row in DATASET.iterrows()]

print(f"Built {len(batch_requests)} batch requests.")
print(f"\nSample custom_id:  {batch_requests[0]['custom_id']}")
print(f"Sample user msg:   {batch_requests[0]['params']['messages'][0]['content'][:120]}...")

### 6. Submit Batch and Save Metadata

Submitting sends all requests to the Anthropic Batch API. Results will be ready
within 24 hours (typically much faster). The batch ID is saved to `batch_jobs/`
so `genai_scope_batchoutput.ipynb` can retrieve results.

In [ ]:
batch = client.messages.batches.create(requests=batch_requests)

print(f"Batch ID:  {batch.id}")
print(f"Status:    {batch.processing_status}")
print(f"Counts:    {batch.request_counts}")

BATCH_DIR.mkdir(exist_ok=True)
metadata = {
    "batch_id": batch.id,
    "model": MODEL,
    "prompt_path": str(PROMPT_PATH),
    "n_records": len(DATASET),
    "include_reasoning": INCLUDE_REASONING,
    "created_at": datetime.now().isoformat(),
    "dataset_ids": DATASET["id"].tolist(),
}
metadata_path = BATCH_DIR / f"{batch.id}.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"\nMetadata saved to {metadata_path}")
print("\nOpen genai_scope_batchoutput.ipynb to retrieve results once the batch is complete.")